# CC3182 – Visión por Computadora
## Laboratorio 10 – Task 2: Análisis Comparativo

---

## Tabla Comparativa

| Métrica | Escenario A | Escenario B |
|---|---|---|
| Modelo | SD 1.5 (base) | LCM Dreamshaper v7 |
| Tipo | Estándar | Destilado (LCD) |
| Pasos de inferencia | 50 | 4 |
| Tiempo de ejecución | 10.39s | 2.82s |
| VRAM máx (GB) | 3.18 | 3.18 |

---

## ¿Por qué LCM genera imágenes coherentes en 4 pasos mientras SD 1.5 no puede?

SD 1.5 modela el proceso generativo como una cadena de Markov de ~1000 pasos de ruido. En inferencia schedulers como DDIM o DPM++ pueden reducirlo a 20–50 pasos, pero cada uno sigue siendo incremental: el modelo predice el ruido ε para remover una cantidad pequeña y calibrada. Si se fuerza a 4 pasos, el scheduler intenta dar saltos de σ (nivel de ruido) para los que el modelo no fue entrenado. Esto produce artefactos severos o ruido inservible. El flujo de tensores no converge en tan pocas iteraciones.

LCM Dreamshaper v7 fue entrenado con Latent Consistency Distillation (LCD), en donde en lugar de aprender a remover ruido paso a paso, el modelo aprende una función de consistencia que mapea directamente cualquier punto ruidoso de la trayectoria a la imagen limpia final. Esto se logra destilando el ODE solver completo de un modelo profesor (Dreamshaper v7) dentro de los pesos del modelo estudiante. El resultado es que cada paso de inferencia equivale conceptualmente a resolver el ODE completo de la cadena, permitiendo convergencia coherente en 4–8 pasos.

---

## Dictamen arquitectónico: ¿Cuál modelo elegir para producción?

Analizando las métricas obtenidas:

- **Tiempo**: LCM completó la inferencia en 2.82s vs 10.39s del modelo estándar, que es una reducción por un factor 3.7 con solo 4 pasos frente a 50. En una API con millones de requests, esta diferencia se traduce directamente en menor costo de cómputo por imagen y mayor throughput por GPU.
- **VRAM**: Ambos modelos consumieron 3.18 GB en la misma GPU (Colab T4). Esto indica que el cuello de botella de memoria no son los pasos de inferencia sino el tamaño de los pesos del modelo y los tensores latentes activos. Sin embargo, la menor latencia de LCM permite liberar la GPU más rápido, aumentando la cantidad de requests concurrentes que se pueden atender.

Para una API en producción con millones de usuarios se elegiría el Escenario B (LCM):
1. **Latencia 3.7x menor** con calidad visual aceptable para la mayoría de casos de uso (previsualizaciones, thumbnails, generación dinámica de contenido).
2. **Mayor throughput**: al liberar la GPU ~3.7x más rápido, se pueden atender más usuarios simultáneos con la misma infraestructura, reduciendo el costo por request.
3. **Escalabilidad horizontal**: con tiempos sub-3s por imagen, el load balancing entre instancias GPU es más eficiente y predecible.

El único escenario donde se elegiría **Escenario A (SD 1.5 estándar)** es la generación de activos finales de alta calidad donde el detalle fotorrealista y la coherencia semántica son críticos — por ejemplo, renders para publicación o concept art profesional. En ese contexto el costo adicional de latencia se justifica con el resultado visual.